In [0]:
# THEME PARK OPERATIONS - DATASET SETUP
# Upload the five data files to the same Unity Catalog Volume folder first.
# Change BASE_PATH, CATALOG, and SCHEMA below to match your workspace.
# WARNING: Rerunning this notebook overwrites the initial training tables.
# Do NOT rerun it after starting the streaming pipeline.

In [0]:
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, DoubleType)

CATALOG = "workspace"      # Change to your catalog
SCHEMA = "themepark"    # Change to your schema
BASE_PATH = "/Volumes/workspace/themepark/vol"  # Change to your folder

prefix = f"{CATALOG}.{SCHEMA}"

entry_schema = StructType([
    StructField("entry_id", IntegerType(), True),
    StructField("visitor_id", StringType(), True),
    StructField("attraction_id", StringType(), True),
    StructField("visit_date", StringType(), True),
    StructField("entry_time", StringType(), True),
    StructField("wait_minutes", IntegerType(), True),
    StructField("ride_minutes", IntegerType(), True),
    StructField("ticket_type", StringType(), True),
    StructField("ticket_price", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("booking_channel", StringType(), True)
])

maintenance_schema = StructType([
    StructField("maintenance_id", StringType(), True),
    StructField("attraction_id", StringType(), True),
    StructField("maintenance_date", StringType(), True),
    StructField("issue_type", StringType(), True),
    StructField("downtime_minutes", IntegerType(), True),
    StructField("maintenance_status", StringType(), True)
])

In [0]:
def read_csv(filename, schema):
    return (spark.read.format("csv")
            .option("header", "true")
            .schema(schema)
            .load(f"{BASE_PATH}/{filename}"))

entries = read_csv("park_entries.csv", entry_schema)
attractions = spark.read.json(f"{BASE_PATH}/attractions.json")
maintenance = read_csv("attraction_maintenance.csv", maintenance_schema)
changes = read_csv("park_entry_changes.csv", entry_schema)

In [0]:
# Source table is append-only for later streaming exercises.
entries.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{prefix}.tp_entries_raw")
attractions.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{prefix}.tp_attractions")
maintenance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{prefix}.tp_maintenance")
changes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{prefix}.tp_entry_changes")

In [0]:
# Inspect the tables. Keep this source unchanged until incremental loading.
for name in ["tp_entries_raw", "tp_attractions", "tp_maintenance", "tp_entry_changes"]:
    print(f"{prefix}.{name}: {spark.table(f'{prefix}.{name}').count()} rows")

display(spark.table(f"{prefix}.tp_entries_raw").limit(10))

workspace.themepark.tp_entries_raw: 121 rows
workspace.themepark.tp_attractions: 10 rows
workspace.themepark.tp_maintenance: 28 rows
workspace.themepark.tp_entry_changes: 10 rows


entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


In [0]:
# LATER, after your streaming pipeline has been created and run once:
# Read park_entries_incremental.csv with entry_schema and append it to
# tp_entries_raw. This is intentionally left for the participant to implement.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA themepark;

**A. Data ingestion and exploration**

01. Load the supplied CSV and JSON datasets using the setup notebook. Display each table and verify its row count.

In [0]:
for t in ["tp_entries_raw","tp_attractions","tp_maintenance","tp_entry_changes"]:
    df = spark.table(t); print(t, df.count()); display(df)

tp_entries_raw 121


entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


tp_attractions 10


attraction_id,attraction_name,capacity_per_hour,category,features,is_active,location,min_height_cm,opened_year
A01,Sky Spinner,140,Thrill,"List(Outdoor, High Speed)",true,"List(N1, North)",120,2018
A02,Coral Carousel,200,Family,"List(Indoor, Family Friendly)",true,"List(E2, East)",0,2015
A03,Jungle Rapids,110,Water,"List(Water Ride, Outdoor)",true,"List(S1, South)",110,2020
A04,Moonlight Train,180,Family,"List(Scenic, Family Friendly)",true,"List(W3, West)",0,2012
A05,Meteor Drop,100,Thrill,"List(High Speed, Outdoor)",true,"List(N2, North)",135,2022
A06,Pirate Voyage,130,Water,"List(Water Ride, Themed)",true,"List(E1, East)",100,2019
A07,Crystal Maze,160,Adventure,"List(Indoor, Puzzle)",true,"List(S3, South)",90,2021
A08,Forest Flyers,120,Adventure,"List(Outdoor, Interactive)",true,"List(W1, West)",115,2023
A09,Galaxy Theater,260,Show,"List(Indoor, Live Show)",true,"List(N3, North)",0,2017
A10,Mini Explorers,190,Family,"List(Kids, Indoor)",true,"List(E3, East)",0,2016


tp_maintenance 28


maintenance_id,attraction_id,maintenance_date,issue_type,downtime_minutes,maintenance_status
M001,A05,2026-10-15,Cleaning,90,Scheduled
M002,A08,2026-10-05,Mechanical,60,Completed
M003,A02,2026-10-12,Mechanical,0,Scheduled
M004,A09,2026-10-18,Electrical,0,Open
M005,A05,2026-10-12,Electrical,0,Scheduled
M006,A04,2026-10-07,Water System,45,Completed
M007,A10,2026-10-20,Mechanical,120,Completed
M008,A04,2026-10-04,Water System,90,Open
M009,A08,2026-10-19,Safety Check,30,Scheduled
M010,A02,2026-10-15,Safety Check,0,Open


tp_entry_changes 10


entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,400.0,Completed,App
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,675.0,Completed,App
1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,500.0,Completed,App
1068,V531,A10,2026-10-15,16:20:00,30,4,Express,725.0,Completed,App
1095,V563,A07,2026-10-09,11:20:00,75,4,Standard,500.0,Completed,App
1121,V700,A01,2026-10-20,11:00:00,15,8,Standard,350.0,Completed,Web
1122,V701,A02,2026-10-20,12:00:00,20,10,Express,600.0,Completed,Web
1123,V702,A03,2026-10-20,13:00:00,25,12,Standard,350.0,Completed,Web
1124,V703,A04,2026-10-20,14:00:00,30,14,Express,600.0,Completed,Web
1125,V704,A05,2026-10-20,15:00:00,35,16,Standard,350.0,Completed,Web


02. Inspect the schema of tp_entries_raw. Identify string, integer, and decimal columns.

In [0]:
raw = spark.table("tp_entries_raw")
raw.printSchema()

root
 |-- entry_id: integer (nullable = true)
 |-- visitor_id: string (nullable = true)
 |-- attraction_id: string (nullable = true)
 |-- visit_date: string (nullable = true)
 |-- entry_time: string (nullable = true)
 |-- wait_minutes: integer (nullable = true)
 |-- ride_minutes: integer (nullable = true)
 |-- ticket_type: string (nullable = true)
 |-- ticket_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- booking_channel: string (nullable = true)



03. Select entry_id, attraction_id, visit_date, ticket_type, and ticket_price. Display the first 20 rows.

In [0]:
display(raw.select("entry_id","attraction_id","visit_date","ticket_type","ticket_price").limit(20))

entry_id,attraction_id,visit_date,ticket_type,ticket_price
1001,A08,2026-10-06,Standard,350.0
1002,A01,2026-10-10,Standard,350.0
1003,A04,2026-10-11,Standard,350.0
1004,A05,2026-10-15,Express,650.0
1005,A07,2026-10-07,Standard,350.0
1006,A08,2026-10-09,VIP,900.0
1007,A08,2026-10-14,Standard,350.0
1008,A06,2026-10-01,Express,650.0
1009,A02,2026-10-16,Standard,350.0
1010,A02,2026-10-09,Express,600.0


04. Using SQL, find Completed entries where ticket_price is greater than 500.

In [0]:
%sql
SELECT * FROM tp_entries_raw WHERE status='Completed' AND ticket_price > 500;

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1014,V564,A07,2026-10-01,14:10:00,65,4,VIP,900.0,Completed,Partner
1018,V535,A10,2026-10-01,13:10:00,65,8,Express,600.0,Completed,App
1021,V527,A03,2026-10-02,13:00:00,-8,8,Express,650.0,Completed,Web
1022,V543,A09,2026-10-15,17:20:00,65,4,Express,600.0,Completed,App
1024,V537,A06,2026-10-06,17:30:00,50,8,Express,650.0,Completed,Web
1025,V513,A09,2026-10-08,14:20:00,65,12,Express,600.0,Completed,App
1028,V547,A08,2026-10-02,13:20:00,75,12,Express,600.0,Completed,App


05. Use DISTINCT and IN to examine ticket types and booking channels.

In [0]:
%sql
SELECT DISTINCT ticket_type FROM tp_entries_raw;

ticket_type
Standard
Express
VIP


In [0]:
%sql
SELECT DISTINCT booking_channel FROM tp_entries_raw WHERE booking_channel IN ('Web','Kiosk','Partner','App');

booking_channel
App
Web
Kiosk
Partner


06. Convert visit_date to a date type and count entries by visit date.

In [0]:
%sql
SELECT to_date(visit_date) AS visit_date, COUNT(*) c FROM tp_entries_raw GROUP BY 1 ORDER BY 1;

visit_date,c
2026-10-01,11
2026-10-02,4
2026-10-03,13
2026-10-04,7
2026-10-05,4
2026-10-06,6
2026-10-07,8
2026-10-08,7
2026-10-09,10
2026-10-10,7


**B. Cleaning and transforming with PySpark**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

07. Find records with missing ticket_price or booking_channel values.

In [0]:
display(raw.filter(F.col("ticket_price").isNull() | F.col("booking_channel").isNull()))

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1016,V563,A10,2026-10-08,15:10:00,30,10,Standard,null,Completed,Kiosk
1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,400.0,Completed,null
1071,V553,A07,2026-10-03,11:10:00,5,20,Standard,null,Completed,Partner
1087,V537,A01,2026-10-11,10:00:00,75,4,Standard,350.0,Completed,null


08. Fill missing booking_channel values with Unknown in a new DataFrame; do not change the raw source.

In [0]:
filled_df = raw.fillna({"booking_channel":"Unknown"})
display(filled_df)

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,Unknown
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


09. Identify records with negative wait_minutes, zero ride_minutes, or nonpositive ticket_price.

In [0]:
bad_df = raw.filter((F.col("wait_minutes")<0)|(F.col("ride_minutes")==0)|(F.col("ticket_price")<=0))
display(bad_df)

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1021,V527,A03,2026-10-02,13:00:00,-8,8,Express,650.0,Completed,Web
1036,V527,A06,2026-10-04,14:40:00,40,0,Standard,400.0,Completed,Kiosk
1058,V516,A02,2026-10-08,19:10:00,30,4,Standard,0.0,Completed,App
1073,V501,A03,2026-10-07,18:50:00,-3,6,Express,650.0,Completed,App


10. Find duplicate entry_id values and create a deduplicated DataFrame.

In [0]:
display(raw.groupBy("entry_id").count().filter("count>1"))
dedup_df = raw.dropDuplicates(["entry_id"])
display(dedup_df)

entry_id,count
1032,2


entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk


11. Identify entry records whose attraction_id is missing from the attraction master.

In [0]:
att = spark.table("tp_attractions")
orphans = raw.join(att, "attraction_id", "left_anti")
display(orphans)

attraction_id,entry_id,visitor_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
A99,1092,V566,2026-10-03,15:40:00,75,20,Standard,400.0,Completed,Kiosk


12. Create wait_category: Short (0-20), Medium (21-45), Long (above 45). Exclude invalid waits.

In [0]:
category_df = dedup_df.filter(F.col("wait_minutes")>=0).withColumn("wait_category",
    F.when(F.col("wait_minutes")<=20,"Short").when(F.col("wait_minutes")<=45,"Medium").otherwise("Long"))
display(category_df)

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel,wait_category
1001,V515,A08,2026-10-06,17:20:00,15,20,Standard,350.0,Cancelled,App,Short
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App,Long
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web,Medium
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web,Short
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk,Short
1006,V549,A08,2026-10-09,10:30:00,10,8,VIP,900.0,Completed,Web,Short
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web,Short
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk,Medium
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null,Short
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk,Short


13. Join entries with attractions and extract location.zone and features from the nested JSON data.

In [0]:
joined_df = dedup_df.join(att, "attraction_id").select(dedup_df["*"], F.col("location.zone").alias("zone"), "features")
display(joined_df)

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel,zone,features
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web,West,"List(Scenic, Family Friendly)"
1021,V527,A03,2026-10-02,13:00:00,-8,8,Express,650.0,Completed,Web,South,"List(Water Ride, Outdoor)"
1030,V557,A10,2026-10-10,18:30:00,5,20,Standard,350.0,Completed,Kiosk,East,"List(Kids, Indoor)"
1044,V548,A04,2026-10-10,11:20:00,65,20,Express,600.0,Pending,App,West,"List(Scenic, Family Friendly)"
1088,V542,A09,2026-10-11,10:50:00,20,15,VIP,900.0,Cancelled,Web,North,"List(Indoor, Live Show)"
1108,V543,A03,2026-10-16,11:00:00,10,4,Express,650.0,Pending,Partner,South,"List(Water Ride, Outdoor)"
1015,V529,A05,2026-10-01,13:40:00,10,15,Standard,400.0,Completed,Kiosk,North,"List(High Speed, Outdoor)"
1022,V543,A09,2026-10-15,17:20:00,65,4,Express,600.0,Completed,App,North,"List(Indoor, Live Show)"
1025,V513,A09,2026-10-08,14:20:00,65,12,Express,600.0,Completed,App,North,"List(Indoor, Live Show)"
1060,V542,A08,2026-10-05,11:20:00,15,12,Standard,350.0,Completed,App,West,"List(Outdoor, Interactive)"


**C. SQL joins and aggregations**

14. Find Completed entries for Standard or Express tickets using WHERE, IN, AND, and BETWEEN.

In [0]:
%sql
SELECT * FROM tp_entries_raw WHERE status='Completed' AND ticket_type IN ('Standard','Express') AND to_date(visit_date) BETWEEN '2026-01-01' AND '2026-12-31';

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1002,V535,A01,2026-10-10,15:00:00,65,6,Standard,350.0,Completed,App
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web
1004,V517,A05,2026-10-15,17:50:00,20,6,Express,650.0,Completed,Web
1005,V537,A07,2026-10-07,16:50:00,5,6,Standard,350.0,Completed,Kiosk
1007,V529,A08,2026-10-14,14:50:00,5,20,Standard,350.0,Completed,Web
1008,V521,A06,2026-10-01,13:40:00,40,6,Express,650.0,Completed,Kiosk
1009,V510,A02,2026-10-16,16:00:00,10,8,Standard,350.0,Completed,null
1013,V506,A09,2026-10-11,10:20:00,15,20,Standard,350.0,Completed,Web
1015,V529,A05,2026-10-01,13:40:00,10,15,Standard,400.0,Completed,Kiosk
1016,V563,A10,2026-10-08,15:10:00,30,10,Standard,null,Completed,Kiosk


15. Calculate total valid ticket revenue and completed-entry counts by attraction.

In [0]:
%sql
SELECT attraction_id, SUM(ticket_price) revenue, COUNT(*) completed_entries
FROM tp_entries_raw WHERE status='Completed' AND ticket_price>0 GROUP BY 1;

attraction_id,revenue,completed_entries
A01,4200.0,9
A04,5450.0,12
A05,5850.0,10
A07,4750.0,9
A08,3600.0,8
A06,4600.0,9
A02,2000.0,5
A09,3200.0,7
A10,6050.0,13
A03,4500.0,8


16. Calculate average waiting time by attraction category, excluding invalid waits.

In [0]:
%sql
SELECT a.category, AVG(e.wait_minutes) avg_wait FROM tp_entries_raw e
JOIN tp_attractions a USING (attraction_id) WHERE e.wait_minutes>=0 GROUP BY 1;

category,avg_wait
Family,31.463414634146343
Thrill,30.5
Show,35.90909090909091
Adventure,33.958333333333336
Water,25.227272727272727


17. Join maintenance records with attractions and calculate completed-maintenance downtime by attraction.

In [0]:
%sql
SELECT a.attraction_id, SUM(m.downtime_minutes) downtime FROM tp_maintenance m
JOIN tp_attractions a USING (attraction_id) WHERE m.maintenance_status='Completed' GROUP BY 1;

attraction_id,downtime
A10,120
A02,240
A09,0
A01,120
A03,30
A07,120
A06,135
A04,75
A08,60


18. Calculate total completed-entry revenue by park zone using the nested location.zone attribute.

In [0]:
%sql
SELECT a.location.zone, SUM(e.ticket_price) revenue FROM tp_entries_raw e
JOIN tp_attractions a USING (attraction_id) WHERE e.status='Completed' GROUP BY 1;

zone,revenue
North,13250.0
East,12650.0
South,9250.0
West,9050.0


**D. Delta table creation and CRUD**

19. Create a clean managed Delta table named tp_entries_work using CTAS or a PySpark DataFrame write.

In [0]:
clean = (raw.dropDuplicates(["entry_id"]).fillna({"booking_channel":"Unknown"})
         .filter("ticket_price > 0 AND wait_minutes >= 0 AND ride_minutes > 0"))
clean.write.format("delta").mode("overwrite").saveAsTable("tp_entries_work") 

20. Create a second Delta table from a PySpark DataFrame using saveAsTable().

In [0]:
clean.limit(50).write.format("delta").mode("overwrite").saveAsTable("tp_entries_work2")

21. Insert two new valid visitor entries into tp_entries_work using SQL.

In [0]:
%sql
SELECT * FROM tp_entries_work LIMIT 1

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,350.0,Completed,Web


In [0]:
%sql
INSERT INTO tp_entries_work VALUES
(9001, 'V535', 'A04', '2026-11-11', '10:30:00', 10, 10, 'Standard', 250, 'Completed', 'App'),
(9002, 'V626', 'A10', '2026-10-14', '10:30:00', 20, 15, 'Express', 450, 'Completed', 'Web');

num_affected_rows,num_inserted_rows
2,2


22. Update the status and ticket price of an existing entry in tp_entries_work.

In [0]:
%sql
UPDATE tp_entries_work SET status='Completed', ticket_price=450 WHERE entry_id='1003';

num_affected_rows
1


23. Delete a Cancelled entry from tp_entries_work and verify the result.

In [0]:
%sql
DELETE FROM tp_entries_work WHERE entry_id = (SELECT entry_id FROM tp_entries_work WHERE status='Cancelled' LIMIT 1);

num_affected_rows
1


24. Use DESCRIBE EXTENDED to identify the working table type and location; explain managed versus external storage.

In [0]:
%sql
DESCRIBE EXTENDED tp_entries_work;

col_name,data_type,comment
entry_id,int,null
visitor_id,string,null
attraction_id,string,null
visit_date,string,null
entry_time,string,null
wait_minutes,int,null
ride_minutes,int,null
ticket_type,string,null
ticket_price,double,null
status,string,null


**E. MERGE / UPSERT and SCD**

25. Inspect tp_entry_changes and separate existing entry IDs from new IDs.

In [0]:
%sql
SELECT c.*, w.entry_id IS NOT NULL AS exists_in_work
FROM tp_entry_changes c LEFT JOIN tp_entries_work w USING (entry_id);

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel,exists_in_work
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,400.0,Completed,App,true
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,675.0,Completed,App,true
1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,500.0,Completed,App,true
1068,V531,A10,2026-10-15,16:20:00,30,4,Express,725.0,Completed,App,true
1095,V563,A07,2026-10-09,11:20:00,75,4,Standard,500.0,Completed,App,true
1121,V700,A01,2026-10-20,11:00:00,15,8,Standard,350.0,Completed,Web,false
1122,V701,A02,2026-10-20,12:00:00,20,10,Express,600.0,Completed,Web,false
1123,V702,A03,2026-10-20,13:00:00,25,12,Standard,350.0,Completed,Web,false
1124,V703,A04,2026-10-20,14:00:00,30,14,Express,600.0,Completed,Web,false
1125,V704,A05,2026-10-20,15:00:00,35,16,Standard,350.0,Completed,Web,false


26. Use SQL MERGE to update matching entries and insert new entries into tp_entries_work (SCD Type 1).

In [0]:
%sql
MERGE INTO tp_entries_work t USING tp_entry_changes s ON t.entry_id = s.entry_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
10,5,0,5


27. Verify which entry IDs were updated and which were inserted.

In [0]:
%sql
DESCRIBE HISTORY tp_entries_work; 

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-11-11T05:14:19.000Z,72240446752486,tsaravanapriya535@gmail.com,MERGE,"Map(predicate -> [""(entry_id#19464 = entry_id#19486)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1421029179937123),4b34b7ab-399b-4fd1-9739-658ff9f82586,1111-041805-vwancx9k-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4331, numTargetBytesRemoved -> 3823, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 2705, materializeSourceTimeMs -> 5, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 1, scanTimeMs -> 906, numTargetRowsUpdated -> 5, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1730)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-11T05:11:00.000Z,72240446752486,tsaravanapriya535@gmail.com,DELETE,"Map(predicate -> [""(entry_id#18853 = scalar-subquery#18830 [])""])",null,List(1421029179937123),8870e8a8-cf71-448b-97c0-50992d44dfcf,1111-041805-vwancx9k-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1192, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 819, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 372)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-11T05:10:23.000Z,72240446752486,tsaravanapriya535@gmail.com,UPDATE,"Map(predicate -> [""(entry_id#18042 = 1003)""])",null,List(1421029179937123),c3e1b5a3-e8d3-421f-bbc7-dda92ed3c5b1,1111-041805-vwancx9k-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3218, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1461, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 3823, rewriteTimeMs -> 1723)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-11T05:08:07.000Z,72240446752486,tsaravanapriya535@gmail.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> true, mode -> Append, clusteringOnWriteStatus -> null)",null,List(1421029179937123),0f162bda-cd2a-46fe-a1f6-0196ab3bc8ee,1111-041805-vwancx9k-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 3150)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-11T04:59:54.000Z,72240446752486,tsaravanapriya535@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""7e470df1-d27e-49ca-927e-5fe16b7ab440"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-cf8a3606-705e-4943-b207-518626ca6ac1"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-469dbd13-33fd-4825-a255-aed9e4741fdb""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when r

28. Create a separate Delta table and perform the same MERGE using the PySpark DeltaTable API.

In [0]:
spark.table("tp_entries_work").limit(0).write.saveAsTable("tp_entries_work_py")
(DeltaTable.forName(spark,"tp_entries_work_py").alias("t")
  .merge(spark.table("tp_entry_changes").alias("s"), "t.entry_id=s.entry_id")
  .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

29. Create an attraction-capacity history table with valid_from, valid_to, and is_current columns.

In [0]:
%sql
CREATE OR REPLACE TABLE tp_capacity_history AS
SELECT attraction_id, capacity_per_hour, current_date() valid_from, CAST(NULL AS DATE) valid_to, true is_current
FROM tp_attractions;

num_affected_rows,num_inserted_rows


30. Implement an SCD Type 2 change for one attraction capacity; query both its historical and current rows.

In [0]:
%sql
UPDATE tp_capacity_history SET valid_to=current_date(), is_current=false
WHERE attraction_id='A01' AND is_current;
INSERT INTO tp_capacity_history VALUES ('A01', 120, current_date(), NULL, true);
SELECT * FROM tp_capacity_history WHERE attraction_id='A01' ORDER BY valid_from;

attraction_id,capacity_per_hour,valid_from,valid_to,is_current
A01,140,2026-11-11,2026-11-11,false
A01,120,2026-11-11,null,true


**F. Delta internals, Time Travel, and optimization**

31. Use DESCRIBE DETAIL and DESCRIBE HISTORY on tp_entries_work. Record the storage format and operations.

In [0]:
%sql
DESCRIBE DETAIL tp_entries_work;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,c08564b8-b65a-4128-8d12-9901bcafc484,workspace.themepark.tp_entries_work,null,,2026-11-11T04:59:51.587Z,2026-11-11T05:14:19.000Z,List(),List(),3,12176,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 7e470df1-d27e-49ca-927e-5fe16b7ab440, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-cf8a3606-705e-4943-b207-518626ca6ac1, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-469dbd13-33fd-4825-a255-aed9e4741fdb)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 6, numDeletionVectors -> 1)",true


In [0]:
%sql
DESCRIBE HISTORY tp_entries_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-11-11T05:14:19.000Z,72240446752486,tsaravanapriya535@gmail.com,MERGE,"Map(predicate -> [""(entry_id#19464 = entry_id#19486)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1421029179937123),4b34b7ab-399b-4fd1-9739-658ff9f82586,1111-041805-vwancx9k-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4331, numTargetBytesRemoved -> 3823, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 2705, materializeSourceTimeMs -> 5, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 1, scanTimeMs -> 906, numTargetRowsUpdated -> 5, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1730)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-11T05:11:00.000Z,72240446752486,tsaravanapriya535@gmail.com,DELETE,"Map(predicate -> [""(entry_id#18853 = scalar-subquery#18830 [])""])",null,List(1421029179937123),8870e8a8-cf71-448b-97c0-50992d44dfcf,1111-041805-vwancx9k-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1192, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 819, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 372)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-11T05:10:23.000Z,72240446752486,tsaravanapriya535@gmail.com,UPDATE,"Map(predicate -> [""(entry_id#18042 = 1003)""])",null,List(1421029179937123),c3e1b5a3-e8d3-421f-bbc7-dda92ed3c5b1,1111-041805-vwancx9k-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3218, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1461, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 3823, rewriteTimeMs -> 1723)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-11T05:08:07.000Z,72240446752486,tsaravanapriya535@gmail.com,WRITE,"Map(partitionBy -> [], clusterBy -> [], statsOnLoad -> true, mode -> Append, clusteringOnWriteStatus -> null)",null,List(1421029179937123),0f162bda-cd2a-46fe-a1f6-0196ab3bc8ee,1111-041805-vwancx9k-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 3150)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-11T04:59:54.000Z,72240446752486,tsaravanapriya535@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""7e470df1-d27e-49ca-927e-5fe16b7ab440"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-cf8a3606-705e-4943-b207-518626ca6ac1"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-469dbd13-33fd-4825-a255-aed9e4741fdb""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when r

32. Find the table version immediately before your MERGE and query it using VERSION AS OF.

In [0]:
%sql
SELECT * FROM tp_entries_work VERSION AS OF 3 WHERE entry_id='1003';

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,450.0,Completed,Web


33. Compare one updated entry before and after the MERGE.

In [0]:
%sql
SELECT * FROM tp_entries_work VERSION AS OF 3 WHERE entry_id='1003';

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,450.0,Completed,Web


In [0]:
%sql
SELECT * FROM tp_entries_work WHERE entry_id='1003';

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,400.0,Completed,App


34. Run OPTIMIZE, then compare the file count before and after. Explain why small tables may not change.

In [0]:
%sql
OPTIMIZE tp_entries_work;

path,metrics
,"List(1, 3, List(5793, 5793, 5793.0, 1, 5793), List(3150, 4695, 4058.6666666666665, 3, 12176), 0, null, null, 0, 1, 3, 0, true, 0, 0, 1794374688304, 1794374690817, 8, 1, null, List(1, 6), null, 11, 11, 438, 0, null, null, 0)"


A small table is already one or a few files, so there is nothing to compact.

35. Run ZORDER BY (attraction_id) if your table is not liquid clustered, and inspect the history.

In [0]:
%sql
OPTIMIZE tp_entries_work ZORDER BY (attraction_id);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 5793), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1794374742758, 1794374743404, 8, 0, null, List(0, 0), null, 11, 11, 0, 0, null, null, 0)"


36. Explain VACUUM retention and perform a safe dry run if supported; do not delete historical files.

The default retention is 7 days. VACUUM deletes unreferenced files older than that, which breaks Time Travel to older versions.

In [0]:
%sql
VACUUM tp_entries_work DRY RUN;

path


**G. Views and Parquet**

37. Create a permanent view of valid Completed entries joined with attraction names.

In [0]:
%sql
CREATE OR REPLACE VIEW v_completed_entries AS
SELECT e.*, a.attraction_name FROM tp_entries_work e JOIN tp_attractions a USING (attraction_id)
WHERE e.status='Completed' AND e.ticket_price>0;

38. Create a temporary view of entries with wait_minutes greater than 45.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tv_long_wait AS SELECT * FROM tp_entries_work WHERE wait_minutes>45;

39. Create and query a global temporary view if supported by your compute.

In [0]:
%sql
CREATE OR REPLACE GLOBAL TEMP VIEW gv_long_wait AS SELECT * FROM tp_entries_work WHERE wait_minutes>45;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6481017297886577>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE OR REPLACE GLOBAL TEMP VIEW gv_long_wait AS SELECT * FROM tp_entries_work WHERE wait_minutes>45;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2565, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2563 with self.builtin_trap:
   2564     args = (magic_arg_s, cell)
-> 2565     result = fn(*args, **kwargs)
   2567 # The code below prevents the output from being displayed
   2568 # when using magics with decorator @output_can_be_silenced
   2569 # when the last Python token in the expression is a ';'.
   2570 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:217, in SqlMagic.sql(self, line, cell)
    210 e

40. Write a DataFrame to Parquet in an approved Volume or external location; read it back and inspect its schema.

In [0]:
p = "/Volumes/workspace/themepark/vol/parquet_output"
spark.table("tp_entries_work").write.mode("overwrite").parquet(p)

41. Convert that Parquet data to a new Delta table by reading it and writing Delta; explain what the Delta transaction log adds.

In [0]:
pq = spark.read.parquet(p)
pq.printSchema()
pq.write.format("delta").mode("overwrite").saveAsTable("tp_entries_from_parquet")

root
 |-- entry_id: integer (nullable = true)
 |-- visitor_id: string (nullable = true)
 |-- attraction_id: string (nullable = true)
 |-- visit_date: string (nullable = true)
 |-- entry_time: string (nullable = true)
 |-- wait_minutes: integer (nullable = true)
 |-- ride_minutes: integer (nullable = true)
 |-- ticket_type: string (nullable = true)
 |-- ticket_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- booking_channel: string (nullable = true)



The Delta transaction log (_delta_log) adds ACID transactions, versioning and Time Travel, schema enforcement, and a reliable record of which files make up the table.

**H. Structured Streaming**

42. Create a streaming DataFrame from Spark rate with five rows per second.

In [0]:
s = spark.readStream.format("rate").option("rowsPerSecond",5).load()

43. Create visitor_event_id and simulated_wait_minutes columns from the streaming value.

In [0]:
s2 = (s.withColumn("visitor_event_id", F.concat(F.lit("V"), F.col("value")))
        .withColumn("simulated_wait_minutes", (F.col("value")*7 % 60).cast("int")))

44. Filter the stream to retain simulated waits above 30 minutes.

In [0]:
f = s2.filter("simulated_wait_minutes > 30")

45. Aggregate streaming events into two wait categories, display results with a supported sink, and stop the query.

In [0]:
agg = (f.withColumn("wait_cat", F.when(F.col("simulated_wait_minutes")<=45,"Medium").otherwise("Long"))
        .withWatermark("timestamp","10 seconds")
        .groupBy(F.window("timestamp","10 seconds"),"wait_cat").count())

In [0]:
q = (agg.writeStream.outputMode("append").trigger(availableNow=True)
     .option("checkpointLocation", "/Volumes/workspace/themepark/vol/ckpt/rate")
     .toTable("tp_stream_out"))

**I. Lakeflow / DLT and incremental loading**

46. Create a SQL Lakeflow pipeline with a Bronze streaming table reading append-only tp_entries_raw.

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE tp_bronze_sql AS
SELECT * FROM STREAM(workspace.themepark.tp_entries_raw);

message
The operation was successfully executed.


47. Create a Silver streaming table that retains valid Completed entries and calculates ticket revenue.

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE tp_silver_sql AS
SELECT *, ticket_price AS revenue FROM STREAM(tp_bronze_sql)
WHERE status='Completed' AND ticket_price>0 AND wait_minutes>=0 AND ride_minutes>0;

message
The operation was successfully executed.


48. Create a Gold materialized view showing entry counts and revenue by attraction; run and inspect the pipeline graph.

In [0]:
%sql
CREATE OR REFRESH MATERIALIZED VIEW tp_gold_sql AS
SELECT attraction_id, COUNT(*) entries, SUM(revenue) revenue FROM tp_silver_sql GROUP BY 1;

result
The operation was successfully executed.


49. Create a separate Python Lakeflow pipeline for the same Bronze-Silver-Gold flow using distinct output names.

In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

# Bronze: streaming table over the append-only raw table
@dp.table(name="tp_bronze_py")
def tp_bronze_py():
    return spark.readStream.table("workspace.themepark.tp_entries_raw")

# Silver: streaming table, valid Completed entries plus revenue
@dp.table(name="tp_silver_py")
def tp_silver_py():
    return (
        spark.readStream.table("tp_bronze_py")
        .filter("status = 'Completed' AND ticket_price > 0 AND wait_minutes >= 0 AND ride_minutes > 0")
        .withColumn("revenue", F.col("ticket_price"))
    )

# Gold: materialized view, batch read from Silver
@dp.materialized_view(name="tp_gold_py")
def tp_gold_py():
    return (
        spark.read.table("tp_silver_py")
        .groupBy("attraction_id")
        .agg(F.count("*").alias("entries"), F.sum("revenue").alias("revenue"))
    )

50. Append park_entries_incremental.csv to tp_entries_raw only after the first pipeline run; rerun the pipeline and verify the Gold totals.

In [0]:
inc = spark.read.csv(f"{BASE_PATH}/park_entries_incremental.csv", header=True, inferSchema=True)
inc.write.mode("append").saveAsTable("tp_entries_raw")

**Final capstone - Park Operations Dashboard**

Build a complete, repeatable visitor-analytics workflow using the supplied datasets. Use the same techniques practiced in the assessment.

• Bronze: ingest the append-only visitor-entry Delta source.

• Silver: validate records, handle missing channels, and calculate revenue.

• Gold: aggregate entries, revenue, and average waiting time by attraction and zone.

• Create a bar chart for revenue by attraction, a pie chart for ticket-type share, a line chart for daily revenue, and a scatter plot comparing average wait and revenue.

• Demonstrate a successful MERGE, one historical table version, and a second incremental pipeline update.

• Explain how you would separate development and production schemas and pipeline configurations.

In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

# BRONZE: raw, append-only source, no changes
@dp.table(name="tp_cap_bronze", comment="Raw visitor entries")
def tp_cap_bronze():
    return spark.readStream.table("workspace.themepark.tp_entries_raw")

# SILVER: validate, dedupe, fill channel, calculate revenue
@dp.table(name="tp_cap_silver", comment="Validated completed entries")
def tp_cap_silver():
    return (
        spark.readStream.table("tp_cap_bronze")
        .filter("status = 'Completed'")
        .filter("ticket_price > 0")
        .filter("wait_minutes >= 0")
        .filter("ride_minutes > 0")
        .filter("attraction_id IS NOT NULL")
        .dropDuplicates(["entry_id"])
        .withColumn("booking_channel", F.coalesce("booking_channel", F.lit("Unknown")))
        .withColumn("visit_date", F.to_date("visit_date"))
        .withColumn("revenue", F.col("ticket_price").cast("double"))
    )

# GOLD: entries, revenue, avg wait by attraction and zone
@dp.materialized_view(name="tp_cap_gold", comment="KPIs by attraction and zone")
def tp_cap_gold():
    s = spark.read.table("tp_cap_silver")
    a = spark.read.table("workspace.themepark.tp_attractions")
    return (
        s.join(a, "attraction_id", "inner")      # inner join also drops orphan IDs
         .groupBy("attraction_id", "attraction_name", F.col("location.zone").alias("zone"))
         .agg(F.count("*").alias("entries"),
              F.round(F.sum("revenue"), 2).alias("total_revenue"),
              F.round(F.avg("wait_minutes"), 1).alias("avg_wait_minutes"))
    )

**Refer Screenshots for the following visualizations.**

**1. Bar chart: revenue by attraction**

In [0]:
%sql
SELECT attraction_name, SUM(total_revenue) AS revenue
FROM workspace.themepark.tp_cap_gold GROUP BY 1 ORDER BY 2 DESC;

attraction_name,revenue
Mini Explorers,6400.0
Meteor Drop,5850.0
Moonlight Train,5200.0
Pirate Voyage,5100.0
Crystal Maze,5100.0
Sky Spinner,4800.0
Forest Flyers,4200.0
Jungle Rapids,4100.0
Coral Carousel,3500.0
Galaxy Theater,3200.0


Databricks visualization. Run in Databricks to view.

**2. Pie chart: ticket-type share**

In [0]:
%sql
SELECT ticket_type, COUNT(*) AS entries, SUM(revenue) AS revenue
FROM workspace.themepark.tp_cap_silver GROUP BY 1;

ticket_type,entries,revenue
Express,30,18400.0
Standard,56,20300.0
VIP,10,9150.0


Databricks visualization. Run in Databricks to view.

**3. Line chart: daily revenue**

In [0]:
%sql
SELECT visit_date, SUM(revenue) AS daily_revenue
FROM workspace.themepark.tp_cap_silver GROUP BY 1 ORDER BY 1;

visit_date,daily_revenue
2026-10-01,5250.0
2026-10-02,1000.0
2026-10-03,3900.0
2026-10-04,2450.0
2026-10-05,1550.0
2026-10-06,1950.0
2026-10-07,2250.0
2026-10-08,2900.0
2026-10-09,2350.0
2026-10-10,1750.0


Databricks visualization. Run in Databricks to view.

**4. Scatter plot: average wait versus revenue**

In [0]:
%sql
SELECT attraction_name, avg_wait_minutes, total_revenue
FROM workspace.themepark.tp_cap_gold;

attraction_name,avg_wait_minutes,total_revenue
Mini Explorers,31.4,6400.0
Pirate Voyage,29.4,5100.0
Coral Carousel,35.0,3500.0
Moonlight Train,32.1,5200.0
Meteor Drop,19.5,5850.0
Galaxy Theater,35.0,3200.0
Jungle Rapids,17.9,4100.0
Forest Flyers,33.9,4200.0
Crystal Maze,35.0,5100.0
Sky Spinner,42.5,4800.0


Databricks visualization. Run in Databricks to view.

**Successful MERGE**

In [0]:
%sql
MERGE INTO workspace.themepark.tp_entries_work t
USING workspace.themepark.tp_entry_changes s
ON t.entry_id = s.entry_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

DESCRIBE HISTORY workspace.themepark.tp_entries_work;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-11-11T06:07:28.000Z,72240446752486,tsaravanapriya535@gmail.com,MERGE,"Map(predicate -> [""(entry_id#46821 = entry_id#46832)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1421029179937123),37227b4e-9c3a-4a0b-8938-5ee5db333e54,1111-041805-vwancx9k-v2n,5,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4337, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 10, executionTimeMs -> 2805, materializeSourceTimeMs -> 1, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 835, numTargetRowsUpdated -> 10, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1929)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
5,2026-11-11T05:24:51.000Z,72240446752486,tsaravanapriya535@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(1421029179937123),b0135aac-67f2-4c05-a8e7-766930db27f5,1111-041805-vwancx9k-v2n,4,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 12176, p25FileSize -> 5793, numDeletionVectorsRemoved -> 1, minFileSize -> 5793, numAddedFiles -> 1, maxFileSize -> 5793, p75FileSize -> 5793, p50FileSize -> 5793, numAddedBytes -> 5793)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-11T05:14:19.000Z,72240446752486,tsaravanapriya535@gmail.com,MERGE,"Map(predicate -> [""(entry_id#19464 = entry_id#19486)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}], clusteringOnWriteStatus -> null)",null,List(1421029179937123),4b34b7ab-399b-4fd1-9739-658ff9f82586,1111-041805-vwancx9k-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 4331, numTargetBytesRemoved -> 3823, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 5, executionTimeMs -> 2705, materializeSourceTimeMs -> 5, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 1, scanTimeMs -> 906, numTargetRowsUpdated -> 5, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 10, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1730)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-11T05:11:00.000Z,72240446752486,tsaravanapriya535@gmail.com,DELETE,"Map(predicate -> [""(entry_id#18853 = scalar-subquery#18830 [])""])",null,List(1421029179937123),8870e8a8-cf71-448b-97c0-50992d44dfcf,1111-041805-vwancx9k-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1192, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 819, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 372)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-11T05:10:23.000Z,72240446752486,tsaravanapriya535@gmail.com,UPDATE,"Map(predicate -> [""(entry_id#18042 = 1003)""])",null,List(1421029179937123),c3e1b5a3-e8d3-421f-bbc7-dda92ed3c5b1,1111-041805-vwancx9k-v2

**One historical version**

In [0]:
%sql
SELECT * FROM workspace.themepark.tp_entries_work VERSION AS OF 5
WHERE entry_id IN (SELECT entry_id FROM workspace.themepark.tp_entry_changes);

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,400.0,Completed,App
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,675.0,Completed,App
1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,500.0,Completed,App
1068,V531,A10,2026-10-15,16:20:00,30,4,Express,725.0,Completed,App
1095,V563,A07,2026-10-09,11:20:00,75,4,Standard,500.0,Completed,App
1121,V700,A01,2026-10-20,11:00:00,15,8,Standard,350.0,Completed,Web
1122,V701,A02,2026-10-20,12:00:00,20,10,Express,600.0,Completed,Web
1123,V702,A03,2026-10-20,13:00:00,25,12,Standard,350.0,Completed,Web
1124,V703,A04,2026-10-20,14:00:00,30,14,Express,600.0,Completed,Web
1125,V704,A05,2026-10-20,15:00:00,35,16,Standard,350.0,Completed,Web


In [0]:
%sql
SELECT * FROM workspace.themepark.tp_entries_work
WHERE entry_id IN (SELECT entry_id FROM workspace.themepark.tp_entry_changes);

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,400.0,Completed,App
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,675.0,Completed,App
1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,500.0,Completed,App
1068,V531,A10,2026-10-15,16:20:00,30,4,Express,725.0,Completed,App
1095,V563,A07,2026-10-09,11:20:00,75,4,Standard,500.0,Completed,App
1121,V700,A01,2026-10-20,11:00:00,15,8,Standard,350.0,Completed,Web
1122,V701,A02,2026-10-20,12:00:00,20,10,Express,600.0,Completed,Web
1123,V702,A03,2026-10-20,13:00:00,25,12,Standard,350.0,Completed,Web
1124,V703,A04,2026-10-20,14:00:00,30,14,Express,600.0,Completed,Web
1125,V704,A05,2026-10-20,15:00:00,35,16,Standard,350.0,Completed,Web


In [0]:
%sql
SELECT 'v5' AS version, COUNT(*) AS rows FROM workspace.themepark.tp_entries_work VERSION AS OF 5
UNION ALL
SELECT 'current', COUNT(*) FROM workspace.themepark.tp_entries_work;

version,rows
v5,120
current,120


In [0]:
%sql
SELECT version, operation,
       operationMetrics['numTargetRowsInserted'] AS inserted,
       operationMetrics['numTargetRowsUpdated']  AS updated
FROM (DESCRIBE HISTORY workspace.themepark.tp_entries_work)
WHERE operation = 'MERGE';

version,operation,inserted,updated
6,MERGE,0,10
4,MERGE,5,5


In [0]:
%sql
SELECT 'v3 (before MERGE)' AS version, COUNT(*) AS row_count
FROM workspace.themepark.tp_entries_work VERSION AS OF 3
UNION ALL
SELECT 'current', COUNT(*) FROM workspace.themepark.tp_entries_work;

version,row_count
v3 (before MERGE),115
current,120


In [0]:
%sql
-- Before the MERGE
SELECT * FROM workspace.themepark.tp_entries_work VERSION AS OF 3
WHERE entry_id IN (SELECT entry_id FROM workspace.themepark.tp_entry_changes)
ORDER BY entry_id;

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,450.0,Completed,Web
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,600.0,Cancelled,Kiosk
1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,400.0,Completed,Unknown
1068,V531,A10,2026-10-15,16:20:00,30,4,Express,600.0,Completed,Partner
1095,V563,A07,2026-10-09,11:20:00,75,4,Standard,350.0,Pending,Partner


In [0]:
%sql
-- After the MERGE
SELECT * FROM workspace.themepark.tp_entries_work
WHERE entry_id IN (SELECT entry_id FROM workspace.themepark.tp_entry_changes)
ORDER BY entry_id;

entry_id,visitor_id,attraction_id,visit_date,entry_time,wait_minutes,ride_minutes,ticket_type,ticket_price,status,booking_channel
1003,V547,A04,2026-10-11,11:00:00,40,15,Standard,400.0,Completed,App
1010,V560,A02,2026-10-09,11:10:00,15,12,Express,675.0,Completed,App
1040,V514,A03,2026-10-01,19:10:00,5,15,Standard,500.0,Completed,App
1068,V531,A10,2026-10-15,16:20:00,30,4,Express,725.0,Completed,App
1095,V563,A07,2026-10-09,11:20:00,75,4,Standard,500.0,Completed,App
1121,V700,A01,2026-10-20,11:00:00,15,8,Standard,350.0,Completed,Web
1122,V701,A02,2026-10-20,12:00:00,20,10,Express,600.0,Completed,Web
1123,V702,A03,2026-10-20,13:00:00,25,12,Standard,350.0,Completed,Web
1124,V703,A04,2026-10-20,14:00:00,30,14,Express,600.0,Completed,Web
1125,V704,A05,2026-10-20,15:00:00,35,16,Standard,350.0,Completed,Web


**Second incremental pipeline update**

In [0]:
%sql
SELECT SUM(entries) AS entries, SUM(total_revenue) AS revenue FROM workspace.themepark.tp_cap_gold;

entries,revenue
95,47450.0


In [0]:
raw_schema = spark.table("workspace.themepark.tp_entries_raw").schema
inc = spark.read.csv("/Volumes/workspace/themepark/vol/park_entries_incremental.csv",
                     header=True, schema=raw_schema)
print(inc.count()) 
inc.write.mode("append").saveAsTable("workspace.themepark.tp_entries_raw")

12


In [0]:
%sql
SELECT COUNT(*), SUM(ticket_price) FROM workspace.themepark.tp_entries_raw
WHERE entry_id IN (SELECT entry_id FROM workspace.themepark.tp_entry_changes) 
  AND status='Completed' AND ticket_price>0 AND wait_minutes>=0 AND ride_minutes>0;

COUNT(*),SUM(ticket_price)
3,1350.0


I would keep development and production in separate schemas (themepark_dev and themepark_prod) with separate pipelines running the same code. Each pipeline gets its target schema and source schema through configuration parameters, so no names are hardcoded. Dev uses a small sample, development mode and manual runs. Prod uses the full data, production mode, scheduled triggers and read-only access for analysts, with only a service principal allowed to write. Code is version-controlled in Git and promoted from dev to prod, and production is never edited by hand.